In [2]:
import  pandas as pd
from rdflib import Graph
from rdflib.plugins.sparql import prepareQuery



## Load triples into graph

In [7]:
g = Graph()

g_ontology = Graph()
g_ontology.parse('./graphs/dance_ontology.ttl', format='turtle')
# Load existing graph
g.parse('./graphs/final_kg.ttl', format='turtle')

print(f"Loaded {len(g)} triples from existing graph")



Loaded 7362 triples from existing graph


In [8]:
queries ="""
prefix dbo:   <http://dbpedia.org/ontology/>

PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT (COUNT(?entity) AS ?entityCount) WHERE { ?entity rdf:type/rdfs:subClassOf* ?obj }

"""
#checks where entity is a type of something which is an subclass of another thing

query = prepareQuery(queries)
result = g.query(query)

pd.DataFrame(result.bindings)

,entityCount
0,2423


In [9]:
# SPARQL queries to type check URIs
# this prints subject predicate object relations

queries ="""
prefix dbo:   <http://dbpedia.org/ontology/>

PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?s ?p ?o
WHERE {
  ?s ?p ?o .
  FILTER NOT EXISTS {
      ?s rdf:type/rdfs:subClassOf* ?domain .
      ?p rdf:type/rdfs:subPropertyOf* ?predicate .
      ?o rdf:type/rdfs:subClassOf* ?range }
  }

"""

query = prepareQuery(queries)
result = g.query(query)

pd.DataFrame(result.bindings)

,o,p,s
0,Light cardiovascular exercise,http://schema.org/name,http://example.org/dance/Light_cardiovascular_...
1,180,http://example.org/dance/minTempoBPM,http://example.org/dance/tempo_record_Latin_da...
2,0.4,http://example.org/dance/hardnessRatio,http://example.org/dance/record_Novelty_and_fa...
3,http://example.org/dance/HealthBenefit,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/dance/Improved_core_strength
4,http://example.org/dance/EventFestival,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/dance/Historical_balls
...,...,...,...
4535,http://www.wikidata.org/entity/Q167126,http://www.w3.org/2002/07/owl#sameAs,http://example.org/dance/Zabumba
4536,Flute,http://schema.org/name,http://example.org/dance/Flute
4537,Varies by dance style,http://schema.org/name,http://example.org/dance/Varies_by_dance_style
4538,Juan Carlos Copes,http://schema.org/name,http://example.org/dance/Juan_Carlos_Copes


# Completness - Hannes

## Find all "nan" values that were not properly filtered out

In [10]:
## check for nodes that are not defined correctly
#Find all nan values that were not properly filtered out
queries = """
PREFIX schema: <https://schema.org/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT DISTINCT ?s ?p ?o
WHERE {
  ?s ?p ?o .
  FILTER (str(?o) = "nan")
}
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)

,o,p,s
0,nan,http://schema.org/description,http://example.org/dance/video_T9Wuh1gH7Ng
1,nan,http://schema.org/description,http://example.org/dance/video_OSa4tPmduk8
2,nan,http://schema.org/description,http://example.org/dance/video_eJPumgqw_3o
3,nan,http://schema.org/description,http://example.org/dance/video_F4kRAdr90hI


## check for nodes that are not defined correctly

In [11]:
# Not consistent and concise. Nutrition is already defined as NutritionInformation and as property, does not need to be denfined again. Also, is not complete
queries = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT DISTINCT ?s
WHERE {
    ?s ?p ?o .
    ?s a owl:Class .
    FILTER NOT EXISTS { ?prop rdfs:domain ?s }
    FILTER NOT EXISTS { ?prop rdfs:range ?s }
}
"""


query = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)

""


# Concisness - Julio
## Intensional concisness

In [12]:
print(g.serialize(format='turtle'))

@prefix dance: <http://example.org/dance/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix schema1: <http://schema.org/> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix wd: <http://www.wikidata.org/entity/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

dance:age_Adults a dance:AgeGroup ;
    schema1:name "Adults"@en .

dance:age_All_ages a dance:AgeGroup ;
    schema1:name "All ages"@en .

dance:age_Kids a dance:AgeGroup ;
    schema1:name "Kids"@en .

dance:age_Seniors a dance:AgeGroup ;
    schema1:name "Seniors"@en .

dance:age_Teens a dance:AgeGroup ;
    schema1:name "Teens"@en .

dance:costume a owl:DatatypeProperty ;
    rdfs:label "costume"@en ;
    rdfs:domain dance:DanceRecord ;
    rdfs:range xsd:string .

dance:culturalSignificance a owl:DatatypeProperty ;
    rdfs:label "cultural significance"@en ;
    rdfs:domain dance:DanceRecord ;
    rdfs:range xsd:string .

dance:difficulty_Adva

- Some recipes are declared as products: <http://kg-course/food-nutrition/recipe/38> a schema:Product ;
- Two classes for nutrition: Nutrition - NutritionInformation (from the ontology graph)

## Extensional concisness

In [14]:
# Concisness of data - check for duplicate recipes by name
q1 = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX dance: <http://example.org/dance/>

SELECT ?recipeName (COUNT(?recipe) AS ?duplicateCount) (GROUP_CONCAT(STRAFTER(STR(?recipe), "dance/"); separator=", ") AS ?recipeIDs)
WHERE {
    ?recipe a dance:DanceRecord ;
            schema:name ?recipeName .
}
GROUP BY ?recipeName
HAVING (COUNT(?recipe) > 1)
ORDER BY DESC(?duplicateCount)
"""
query  = prepareQuery(q1)
result = g.query(query)
pd.DataFrame(result.bindings)

""


In [22]:
# Concisness of data - different instructions for the same recipe
q2 = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX dance: <http://example.org/dance/>


SELECT ?recipe1 ?recipe2 ?recipeName
WHERE {
    ?recipe1 a dance:DanceRecord ;
             schema:name ?recipeName ;
             schema:uploadDate ?instruction .
             
    ?recipe2 a dance:DanceRecord ;
             schema:name ?recipeName ;
             schema:uploadDate ?instruction .
             
    FILTER (?recipe1 != ?recipe2)
    FILTER (STR(?recipe1) < STR(?recipe2)) 
}
"""
query  = prepareQuery(q2)
result = g.query(query)
pd.DataFrame(result.bindings)

""


In [10]:
# Concisness of data - multiple values for the same nutrition property 

#TODO: Change this to dance data
q3 = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT ?nutritionNode ?property (COUNT(?value) AS ?valueCount)
WHERE {
    ?nutritionNode ?property ?value .
    
    FILTER(?property IN (schema:calories, schema:carbohydrateContent, schema:fatContent, schema:proteinContent))
}
GROUP BY ?nutritionNode ?property
HAVING (COUNT(?value) > 1)
ORDER BY DESC(?valueCount)
"""
query  = prepareQuery(q3)
result = g.query(query)
pd.DataFrame(result.bindings)

,nutritionNode,property,valueCount
0,http://kg-course/food-nutrition/recipe/48/nutr...,https://schema.org/calories,6
1,http://kg-course/food-nutrition/recipe/48/nutr...,https://schema.org/fatContent,6
2,http://kg-course/food-nutrition/recipe/48/nutr...,https://schema.org/carbohydrateContent,6
3,http://kg-course/food-nutrition/recipe/48/nutr...,https://schema.org/proteinContent,6
4,http://kg-course/food-nutrition/recipe/42/nutr...,https://schema.org/fatContent,4
5,http://kg-course/food-nutrition/recipe/42/nutr...,https://schema.org/proteinContent,4
6,http://kg-course/food-nutrition/recipe/42/nutr...,https://schema.org/calories,4
7,http://kg-course/food-nutrition/recipe/42/nutr...,https://schema.org/carbohydrateContent,4


# Consistency - Hannes


## find triples where a property is used on a subject whose type does not match the propertys declared domain

In [18]:
queries = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT DISTINCT ?s ?p ?o ?expectedDomain ?actualType
WHERE {
  ?p rdfs:domain ?expectedDomain .
  ?s ?p ?o .
  ?s rdf:type ?actualType .
  FILTER (?actualType != ?expectedDomain)
  FILTER NOT EXISTS {
    ?actualType rdfs:subClassOf* ?expectedDomain .
  }
}
ORDER BY ?p
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)


,actualType,expectedDomain,o,p,s
0,http://example.org/dance/MusicGenre,http://example.org/dance/DanceStyle,http://example.org/dance/video_8n29Pc4Ik4s,http://example.org/dance/hasYTLink,http://example.org/dance/Bachata
1,http://example.org/dance/MusicGenre,http://example.org/dance/DanceStyle,http://example.org/dance/video_gtQOIc_mp-0,http://example.org/dance/hasYTLink,http://example.org/dance/Bolero
2,http://example.org/dance/MusicGenre,http://example.org/dance/DanceStyle,http://example.org/dance/video_3XBJevNfZMg,http://example.org/dance/hasYTLink,http://example.org/dance/Cha_Cha
3,http://example.org/dance/MusicGenre,http://example.org/dance/DanceStyle,http://example.org/dance/video_3UXtxn8B2OU,http://example.org/dance/hasYTLink,http://example.org/dance/Duranguense
4,http://example.org/dance/MusicGenre,http://example.org/dance/DanceStyle,http://example.org/dance/video_vudZL4_uqLo,http://example.org/dance/hasYTLink,http://example.org/dance/Forró
5,http://example.org/dance/MusicGenre,http://example.org/dance/DanceStyle,http://example.org/dance/video_8nhqnxirY9o,http://example.org/dance/hasYTLink,http://example.org/dance/Lambada
6,http://example.org/dance/MusicGenre,http://example.org/dance/DanceStyle,http://example.org/dance/video_daaHi0jtHlw,http://example.org/dance/hasYTLink,http://example.org/dance/Merengue
7,http://example.org/dance/MusicGenre,http://example.org/dance/DanceStyle,http://example.org/dance/video_-UqAnPyFK4M,http://example.org/dance/hasYTLink,http://example.org/dance/Milonga
8,http://example.org/dance/MusicGenre,http://example.org/dance/DanceStyle,http://example.org/dance/video_rrdKszMyWQk,http://example.org/dance/hasYTLink,http://example.org/dance/Pasodoble
9,http://example.org/dance/MusicGenre,http://example.org/dance/DanceStyle,http://example.org/dance/video_qbPzvHAGvm4,http://example.org/dance/hasYTLink,http://example.org/dance/Zouk



## Find ObjectProperties used with literal values instead of URIs, similar to how its done in completness check

In [19]:

# found that object Property is
queries = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT DISTINCT ?objProp
WHERE {
  ?objProp a owl:ObjectProperty
}
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)

,objProp
0,http://example.org/dance/hasAgeGroup
1,http://example.org/dance/hasAssociatedMusicGenre
2,http://example.org/dance/hasDanceFormation
3,http://example.org/dance/hasDanceStyle
4,http://example.org/dance/hasDanceType
5,http://example.org/dance/hasEventFestival
6,http://example.org/dance/hasFamousPractitioner
7,http://example.org/dance/hasHealthBenefit
8,http://example.org/dance/hasInstrument
9,http://example.org/dance/hasLearningDifficulty


In [20]:
queries = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT DISTINCT ?subject ?property ?object
WHERE {
  ?property a owl:ObjectProperty .
  ?subject ?property ?object .
  FILTER (isLiteral(?object))
}
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)

""



## entities of type schema:Recipe that are missing key expected properties (schema:name, schema:nutrition, schema:recipeIngredient)

In [21]:
#TODO: Finish this

queries = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX schema: <https://schema.org/>

SELECT DISTINCT ?recipe ?missingProperty
WHERE {
  ?recipe rdf:type schema:Recipe .
  VALUES ?missingProperty { schema:name schema:nutrition schema:recipeIngredient schema:recipeInstructions }
  FILTER NOT EXISTS { ?recipe ?missingProperty ?any }
}
ORDER BY ?recipe ?missingProperty
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)


""


# Syntactic Validity - Hannes

## Find all literal Type mismatches (except for langString and String which are passable (I think))

In [25]:

## all cases where the literalType is not the same as the range

queries = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX schema: <https://schema.org/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX dbr: <http://dbpedia.org/resource/>

SELECT DISTINCT ?s ?literal ?range ?literalType
WHERE {
  ?s ?p ?literal .
  ?p rdfs:range ?range .
  FILTER (isLiteral(?literal) && ?p != rdfs:label)

  BIND (datatype(?literal) AS ?literalType)
  FILTER (?range != ?literalType)
  FILTER (!(?range = xsd:string && ?literalType = rdf:langString))
  FILTER (!(?range = rdf:langString && ?literalType = xsd:string))
}
"""
query  = prepareQuery(queries)
result = g.query(query)

pd.DataFrame(result.bindings)


,literal,literalType,range,s
0,PT6M3S,http://www.w3.org/2001/XMLSchema#string,http://www.w3.org/2001/XMLSchema#duration,http://example.org/dance/video_-IdY7BQTCWs
1,PT7M3S,http://www.w3.org/2001/XMLSchema#string,http://www.w3.org/2001/XMLSchema#duration,http://example.org/dance/video_-JCJUaoRPLI
2,PT6M52S,http://www.w3.org/2001/XMLSchema#string,http://www.w3.org/2001/XMLSchema#duration,http://example.org/dance/video_-UqAnPyFK4M
3,PT7M47S,http://www.w3.org/2001/XMLSchema#string,http://www.w3.org/2001/XMLSchema#duration,http://example.org/dance/video_1X8v6rbe27s
4,PT19M34S,http://www.w3.org/2001/XMLSchema#string,http://www.w3.org/2001/XMLSchema#duration,http://example.org/dance/video_1ijkYKoig1o
...,...,...,...,...
119,PT15M45S,http://www.w3.org/2001/XMLSchema#string,http://www.w3.org/2001/XMLSchema#duration,http://example.org/dance/video_wucZxx-XaVA
120,PT4M4S,http://www.w3.org/2001/XMLSchema#string,http://www.w3.org/2001/XMLSchema#duration,http://example.org/dance/video_y-txQrkv1bE
121,PT8M8S,http://www.w3.org/2001/XMLSchema#string,http://www.w3.org/2001/XMLSchema#duration,http://example.org/dance/video_y7_cQiktu7Y
122,PT4M20S,http://www.w3.org/2001/XMLSchema#string,http://www.w3.org/2001/XMLSchema#duration,http://example.org/dance/video_yKx5AW7doqw


## numbers stored as strings

In [26]:
queries = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT DISTINCT ?s ?p ?o
WHERE {
  ?s ?p ?o .
  ?p rdfs:range ?range .
  FILTER (?range IN (xsd:double, xsd:integer, xsd:float, xsd:decimal))
  FILTER (datatype(?o) = xsd:string)
}
ORDER BY ?p
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)

""


## string literals without a language

In [27]:
queries = """
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT DISTINCT ?s ?p ?o
WHERE {
  ?s ?p ?o .
  FILTER (isLiteral(?o))
  FILTER (datatype(?o) = xsd:string)
}
ORDER BY ?p
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)


,o,p,s
0,Madonna - Vogue Choreographed by Dean Lee Twit...,http://schema.org/description,http://example.org/dance/video_nQcYWBrV10U
1,Let Dancing with The Stars Pro Dancer Brian Fo...,http://schema.org/description,http://example.org/dance/video_3XBJevNfZMg
2,This tutorial video is specifically designed t...,http://schema.org/description,http://example.org/dance/video_1X8v6rbe27s
3,"Центр трайбл-культуры в Санкт-Петербурге, колл...",http://schema.org/description,http://example.org/dance/video_MFZG49yVtco
4,Learn how to dance a crazy good Jive Basic wit...,http://schema.org/description,http://example.org/dance/video_KGYQK-8WUPI
...,...,...,...
486,positive,http://www.wikidata.org/entity/Q28659447,http://example.org/dance/video_HZruz8wO0g8
487,positive,http://www.wikidata.org/entity/Q28659447,http://example.org/dance/video_Thnvgf_3aoQ
488,positive,http://www.wikidata.org/entity/Q28659447,http://example.org/dance/video_BzrA38NiyoA
489,positive,http://www.wikidata.org/entity/Q28659447,http://example.org/dance/video_yKx5AW7doqw


# Semantic Accuracy - Mikolaj

## hardnessRatio must be between 0.0 and 1.0

In [40]:
queries = """
PREFIX dance: <http://example.org/dance/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?record ?ratio
WHERE {
  ?record a dance:DanceRecord ;
          dance:hardnessRatio ?ratio .
  FILTER (?ratio < 0.0 || ?ratio > 1.0)
}
ORDER BY ?ratio
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)


""


## 2. minTempoBPM must be strictly less than maxTempoBPM

In [42]:
queries = """
PREFIX dance: <http://example.org/dance/>

SELECT ?tempoRange ?min ?max
WHERE {
  ?tempoRange a dance:TempoRange ;
              dance:minTempoBPM ?min ;
              dance:maxTempoBPM ?max .
  FILTER (?min >= ?max)
}
ORDER BY ?tempoRange
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)


""


## Each DanceRecord must link to exactly one DanceStyle

In [41]:
queries = """
PREFIX dance: <http://example.org/dance/>

SELECT ?record (COUNT(?style) AS ?styleCount)
WHERE {
  ?record a dance:DanceRecord .
  OPTIONAL { ?record dance:hasDanceStyle ?style }
}
GROUP BY ?record
HAVING (COUNT(?style) != 1)
ORDER BY ?styleCount
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)


DanceRecords without exactly one DanceStyle: 0


""


## Each DanceRecord must link to exactly one DanceType

In [43]:
queries = """
PREFIX dance: <http://example.org/dance/>

SELECT ?record (COUNT(?danceType) AS ?typeCount)
WHERE {
  ?record a dance:DanceRecord .
  OPTIONAL { ?record dance:hasDanceType ?danceType }
}
GROUP BY ?record
HAVING (COUNT(?danceType) != 1)
ORDER BY ?typeCount
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)


DanceRecords without exactly one DanceType: 0


""


## Each DanceRecord must link to exactly one LearningDifficulty

In [44]:
queries = """
PREFIX dance: <http://example.org/dance/>

SELECT ?record (COUNT(?diff) AS ?diffCount)
WHERE {
  ?record a dance:DanceRecord .
  OPTIONAL { ?record dance:hasLearningDifficulty ?diff }
}
GROUP BY ?record
HAVING (COUNT(?diff) != 1)
ORDER BY ?diffCount
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)


""


## LearningDifficulty values must only be Beginner, Intermediate or Advanced

In [45]:
queries = """
PREFIX dance: <http://example.org/dance/>
PREFIX schema1: <http://schema.org/>

SELECT DISTINCT ?record ?diffLabel
WHERE {
  ?record a dance:DanceRecord ;
          dance:hasLearningDifficulty ?diff .
  ?diff schema1:name ?diffLabel .
  FILTER (?diffLabel NOT IN ("Beginner"@en, "Intermediate"@en, "Advanced"@en))
}
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)


""


## Each DanceRecord must link to exactly one AgeGroup

In [46]:
queries = """
PREFIX dance: <http://example.org/dance/>

SELECT ?record (COUNT(?age) AS ?ageCount)
WHERE {
  ?record a dance:DanceRecord .
  OPTIONAL { ?record dance:hasAgeGroup ?age }
}
GROUP BY ?record
HAVING (COUNT(?age) != 1)
ORDER BY ?ageCount
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)


DanceRecords without exactly one AgeGroup: 0


""


## AgeGroup values must only be from the defined set

In [47]:
queries = """
PREFIX dance: <http://example.org/dance/>
PREFIX schema1: <http://schema.org/>

SELECT DISTINCT ?record ?ageLabel
WHERE {
  ?record a dance:DanceRecord ;
          dance:hasAgeGroup ?age .
  ?age schema1:name ?ageLabel .
  FILTER (?ageLabel NOT IN ("Kids"@en, "Teens"@en, "Adults"@en, "Seniors"@en, "All ages"@en))
}
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)


""


## Each DanceRecord must have exactly one TempoRange, and that TempoRange must have both min and max BPM

In [48]:
queries = """
PREFIX dance: <http://example.org/dance/>

SELECT ?record ?tempo ?hasMIn ?hasMax
WHERE {
  ?record a dance:DanceRecord ;
          dance:hasTempoRange ?tempo .
  BIND(EXISTS { ?tempo dance:minTempoBPM ?min } AS ?hasMIn)
  BIND(EXISTS { ?tempo dance:maxTempoBPM ?max } AS ?hasMax)
  FILTER (!?hasMIn || !?hasMax)
}
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)


TempoRanges missing minBPM or maxBPM: 0


""


## A DanceRecord's DanceStyle must itself exist as a typed DanceStyle node

In [49]:
queries = """
PREFIX dance: <http://example.org/dance/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?record ?style
WHERE {
  ?record a dance:DanceRecord ;
          dance:hasDanceStyle ?style .
  FILTER NOT EXISTS { ?style a dance:DanceStyle }
}
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)


""


## A DanceRecord's DanceType must itself exist as a typed DanceType node

In [51]:
queries = """
PREFIX dance: <http://example.org/dance/>

SELECT ?record ?danceType
WHERE {
  ?record a dance:DanceRecord ;
          dance:hasDanceType ?danceType .
  FILTER NOT EXISTS { ?danceType a dance:DanceType }
}
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)


""


## Every DanceStyle node must be referenced by at least one DanceRecord

In [52]:
queries = """
PREFIX dance: <http://example.org/dance/>

SELECT ?style
WHERE {
  ?style a dance:DanceStyle .
  FILTER NOT EXISTS { ?record dance:hasDanceStyle ?style }
}
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)


""


## YTLINK nodes must be connected to a DanceRecord via hasYTLink

In [53]:
queries = """
PREFIX dance: <http://example.org/dance/>

SELECT ?ytlink
WHERE {
  ?ytlink a dance:YTLINK .
  FILTER NOT EXISTS { ?record dance:hasYTLink ?ytlink }
}
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)


YTLINK nodes not linked to any DanceRecord (orphans): 0


""


## YTLINK view count and likes must not be negative

In [54]:
queries = """
PREFIX dance: <http://example.org/dance/>
PREFIX schema1: <http://schema.org/>

SELECT ?ytlink ?prop ?val
WHERE {
  ?ytlink a dance:YTLINK .
  ?ytlink ?prop ?val .
  FILTER (?prop IN (schema1:interactionCount, schema1:UserLikes))
  FILTER (?val < 0)
}
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)


""


## Each DanceRecord must have at least one Instrument, MusicGenre, and HealthBenefit

In [55]:
queries = """
PREFIX dance: <http://example.org/dance/>

SELECT ?record
       (COUNT(DISTINCT ?instrument) AS ?instruments)
       (COUNT(DISTINCT ?genre)      AS ?genres)
       (COUNT(DISTINCT ?benefit)    AS ?benefits)
WHERE {
  ?record a dance:DanceRecord .
  OPTIONAL { ?record dance:hasInstrument ?instrument }
  OPTIONAL { ?record dance:hasAssociatedMusicGenre ?genre }
  OPTIONAL { ?record dance:hasHealthBenefit ?benefit }
}
GROUP BY ?record
HAVING (COUNT(DISTINCT ?instrument) = 0 || COUNT(DISTINCT ?genre) = 0 || COUNT(DISTINCT ?benefit) = 0)
ORDER BY ?record
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)


""


# Interoperability - Hannes

## missing domain

In [29]:
queries = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT DISTINCT ?p ?propertyType
WHERE {
  ?p a ?propertyType .
  VALUES ?propertyType { owl:ObjectProperty owl:DatatypeProperty }
  FILTER NOT EXISTS { ?p rdfs:domain ?any }
}
ORDER BY ?propertyType ?p
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)


""


## missing range

In [30]:
queries = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT DISTINCT ?p ?propertyType
WHERE {
  ?p a ?propertyType .
  VALUES ?propertyType { owl:ObjectProperty owl:DatatypeProperty }
  FILTER NOT EXISTS { ?p rdfs:range ?any }
}
ORDER BY ?propertyType ?p
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)


""


## missing label (class)

In [31]:
queries = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT DISTINCT ?c
WHERE {
  ?c a owl:Class .
  FILTER NOT EXISTS { ?c rdfs:label ?any }
}
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)

""


## missing label (property)

In [32]:
queries = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT DISTINCT ?p ?propertyType
WHERE {
  ?p a ?propertyType .
  VALUES ?propertyType { owl:ObjectProperty owl:DatatypeProperty }
  FILTER NOT EXISTS { ?p rdfs:label ?any }
}
ORDER BY ?propertyType ?p
"""
query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)


""


## properties used that are not defined in the ontology


In [33]:
# merge KG and ontology into one graph
g_merged = g + g_ontology

queries = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT DISTINCT ?p
WHERE {
  ?s ?p ?o .
  FILTER (?p != rdf:type)
  FILTER NOT EXISTS { ?p a owl:ObjectProperty }
  FILTER NOT EXISTS { ?p a owl:DatatypeProperty }
  FILTER NOT EXISTS { ?p a owl:AnnotationProperty }
  FILTER NOT EXISTS { ?p a rdf:Property }
}
ORDER BY ?p
"""
query  = prepareQuery(queries)
result = g_merged.query(query)
pd.DataFrame(result.bindings)


,p
0,http://schema.org/description
1,http://schema.org/name
2,http://www.w3.org/2000/01/rdf-schema#comment
3,http://www.w3.org/2000/01/rdf-schema#domain
4,http://www.w3.org/2000/01/rdf-schema#label
5,http://www.w3.org/2000/01/rdf-schema#range
6,http://www.w3.org/2000/01/rdf-schema#subClassOf
7,http://www.w3.org/2002/07/owl#sameAs
